# Integrated-circuit reconstructions

Compare rPIE, LSQML, GM-rPIE, and GM-MAGPIE after 100 epochs with batch 29. These are the canonical real-data reruns used in the paper.

Reconstruction quality is assessed visually from the object images; no ground-truth error is defined. Viewing the included results requires neither the measured HDF5 files nor a GPU.

In [ ]:
STUDY = "real"
DATASET = "integrated_circuit"
RUN_RECONSTRUCTIONS = False


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from magpie_publication import load_index, select_cases, read_history, run_study

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "results" / "index.json").is_file() and (p / "pyproject.toml").is_file())
index = load_index(root=ROOT)
endpoint_csv = ROOT / index["studies"][STUDY]["endpoint_csv"]
endpoint_table = pd.read_csv(endpoint_csv)


In [ ]:
METHODS = ["rpie", "lsqml", "gm_rpie", "gm_magpie"]
LABELS = dict(zip(METHODS, ["rPIE", "LSQML", "GM-rPIE", "GM-MAGPIE"]))
COLORS = dict(zip(METHODS, ["#4477AA", "#EE7733", "#228833", "#AA3377"]))
PARAMETER_LABELS = {"object_alpha": "α_O", "probe_alpha": "α_P", "damping": "damping", "sigma": "σ", "beta": "β"}

def case_table(cases):
    rows = []
    for case in sorted(cases, key=lambda c: (c.get("batch_size") or 0, c.get("eta") or 0, METHODS.index(c["method"]))):
        parameters = ", ".join(f"{PARAMETER_LABELS.get(k, k)}={v:g}" for k, v in case["parameters"].items())
        rows.append({"Method": LABELS[case["method"]], "Parameters": parameters,
                     "η": case.get("eta"), "Batch": case["batch_size"],
                     "Epochs": case["epochs_completed"], "Status": case["status"],
                     "Object NMSE": case.get("object_nmse")})
    return pd.DataFrame(rows).dropna(axis=1, how="all")


## Recorded method settings

In [ ]:
cases = select_cases(STUDY, dataset=DATASET, selected=True, root=ROOT)
assert len(cases) == 4
display(case_table(cases))
histories = {case["method"]: read_history(case, root=ROOT) for case in cases}


## Object reconstructions

Rows show magnitude and phase; columns compare the four methods using shared display settings.

In [ ]:
display(Image(filename=str(ROOT / "results" / "figures" / "numerical_real_integrated_circuit.png")))


For reconstruction reruns, supply the original files following the [measured-data instructions](../../assets/ptycho_real_data/README.md). The archived complex fields and histories are listed in [the result index](../../results/index.json).

## Optional reconstruction rerun

Set `RUN_RECONSTRUCTIONS = True` to repeat this study on a CUDA GPU. Rerun outputs are separate from the archived publication records displayed above.

In [ ]:
if RUN_RECONSTRUCTIONS:
    rerun = run_study("real", dataset=DATASET, root=ROOT)
    print("Rerun exit code:", rerun.returncode)
else:
    print("Rerun disabled; the included publication results are shown above.")
